In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

print('module imported perfectly')

module imported perfectly


In [2]:
df= pd.read_csv('train.csv')

In [3]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
data = np.array(df)
m, n = data.shape
np.random.shuffle(data) # shuffle before splitting into dev and training sets

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

data_train = data[1000:m].T
Ytrain = data_train[0]
Xtrain = data_train[1:n]
Xtrain = Xtrain / 255.
_,m_train = Xtrain.shape

In [5]:

Ytrain

array([3, 2, 9, ..., 7, 8, 9], shape=(41000,))

In [6]:
def parameters():
    w1= np.random.rand(28,784)- 0.5
    b1= np.random.rand(28,1)- 0.5
    w2= np.random.rand(28,28)- 0.5
    b2= np.random.rand(28,1)- 0.5
    w3= np.random.rand(10,28)- 0.5
    b3= np.random.rand(10,1)-0.5
    return w1,b1,w2,b2,w3,b3

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A

def ReLU_deriv(Z):
    return Z > 0

def conversion(Y):
    conversion_Y = np.zeros((Y.size, Y.max() + 1))
    conversion_Y[np.arange(Y.size), Y] = 1
    conversion_Y = conversion_Y.T
    return conversion_Y

In [7]:
#Forward Propogation
def forward_propogation(w1, b1, w2, b2, w3, b3, X):
    z1= w1.dot(X)+ b1
    a1= ReLU(z1)
    z2= w2.dot(a1)+ b2
    a2= ReLU(z2)
    z3= w3.dot(a2)+ b3
    a3= ReLU(z3)
    return z1,a1,z2,a2,z3,a3

#Backward Propogation
def back_propogation(z1,a1,z2,a2,z3,a3,w1,w2,w3, X, Y):
    conversion_Y = conversion(Y)
    dz3= a3- conversion_Y
    dw3= (dz3.dot(a2.T))/m
    db3= (np.sum(dz3))/m
    dz2= (w3.T.dot(dz3))*ReLU_deriv(z2)
    dw2= (dz2.dot(a1.T))/m
    db2= (np.sum(dz2))/m
    dz1= (w2.T.dot(dz2))*ReLU_deriv(z1)
    dw1= (dz1.dot(X.T))/m
    db1= (np.sum(dz1))/m
    return dw1,db1,dw2,db2,dw3,db3

In [8]:
#Updating parameters
def update_params(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, alpha):
    w1= w1- alpha*(dw1)
    b1= b1- alpha*(db1)
    w2= w2- alpha*(dw2)
    b2= b2- alpha*(db2)
    w3= w3- alpha*(dw3)
    b3= b3- alpha*(db3)
    return w1, b1, w2, b2, w3, b3

In [9]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    w1, b1, w2, b2, w3, b3 = parameters()
    for i in range(iterations):
        z1, a1, z2, a2, z3, a3 = forward_propogation(w1, b1, w2, b2, w3, b3, X)
        dw1, db1, dw2, db2, dw3, db3 = back_propogation(z1, a1, z2, a2, z3, a3, w1, w2, w3, X, Y)
        w1, b1, w2, b2, w3, b3 = update_params(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, alpha)
        if i % 15 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(a3)
            print(get_accuracy(predictions, Y))
    return w1, b1, w2, b2, w3, b3

In [10]:
w1, b1, w2, b2, w3, b3 = gradient_descent(Xtrain, Ytrain, 0.1, 700)

Iteration:  0
[16 22 22 ...  4 22  3] [3 2 9 ... 7 8 9]
0.03802439024390244
Iteration:  15
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.059804878048780485
Iteration:  30
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.053
Iteration:  45
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.0508780487804878
Iteration:  60
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05119512195121951
Iteration:  75
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05134146341463414
Iteration:  90
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05117073170731707
Iteration:  105
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05192682926829268
Iteration:  120
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05341463414634146
Iteration:  135
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05475609756097561
Iteration:  150
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.056195121951219514
Iteration:  165
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.05834146341463415
Iteration:  180
[0 0 0 ... 4 0 3] [3 2 9 ... 7 8 9]
0.06053658536585366
Iteration:  195
[6 0 6 ... 4 6 3] [3 2 9 ... 7 8 9]
0.06268292682926829
Iterati

In [11]:
def make_predictions(X, w1, b1, w2, b2):
    _, _, _, A2 = forward_propogation(w1, b1, w2, b2, X)
    predictions = get_predictions(A2)
    return predictions

def test_prediction(index, w1, b1, w2, b2):
    current_image = Xtrain[:, index, None]
    prediction = make_predictions(Xtrain[:, index, None], w1, b1, w2, b2)
    label = Ytrain[index]
    print("Prediction: ", prediction)
    print("Label: ", label)
    
    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

In [12]:
test_prediction(5, w1, b1, w2, b2)

TypeError: forward_propogation() missing 2 required positional arguments: 'b3' and 'X'